# PDF OCR Extractor v3 - High Accuracy Edition

This notebook extracts detailed information from **page 4** of a PDF file with:
- **8 preprocessing variants** tested for optimal OCR
- **Multi-strategy OCR** with automatic best-config selection
- **Strict checkbox detection** (target: 50-100 actual checkboxes)
- **85-95% OCR confidence** target
- Downloadable .txt and .docx output files

**Key Improvements over v1:**
- Better Tesseract configuration with `preserve_interword_spaces`
- Fallback extraction for missed fields
- Stricter checkbox filtering (18-55px, square-ish only)

## Step 1: Install Required Libraries

In [ ]:
# Install required packages with BETTER Tesseract setup
print("📦 Installing packages...")

!pip install -q pdf2image pillow opencv-python pytesseract python-docx

# Install Tesseract with BEST language data
!apt-get update -qq
!apt-get install -y -qq poppler-utils
!apt-get install -y -qq tesseract-ocr
!apt-get install -y -qq tesseract-ocr-eng  # English language data (best)
!apt-get install -y -qq libtesseract-dev

# Set Tesseract path (Colab default)
import os
os.environ['TESSDATA_PREFIX'] = '/usr/share/tesseract-ocr/4.00/tessdata/'

print("✅ All packages installed!")
print("   Tesseract version:", end=" ")
!tesseract --version | head -1

## Step 2: Import Libraries

In [ ]:
import cv2
import numpy as np
import pytesseract
from pdf2image import convert_from_path
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt
from docx import Document
from docx.shared import Pt, RGBColor
from datetime import datetime
import io
from google.colab import files
import os

# Configure Tesseract path for Colab
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

print("✅ All libraries imported!")

## Step 3: Upload PDF File

In [ ]:
print("📤 Please upload your PDF file...")
uploaded = files.upload()

# Get the uploaded filename
pdf_filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {pdf_filename}")

## Step 4: Convert PDF Page 4 to High-Resolution Image

In [ ]:
print("\n📄 Converting PDF page 4 to high-resolution image...")

# Convert only page 4 with HIGH DPI for better OCR
pages = convert_from_path(pdf_filename, dpi=400, first_page=4, last_page=4)

if len(pages) == 0:
    print("❌ Error: Page 4 not found in PDF!")
else:
    page_4_image = pages[0]
    
    # Convert PIL Image to numpy array for OpenCV
    image_np = np.array(page_4_image)
    
    # Convert RGB to BGR (OpenCV format)
    image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
    
    print(f"✅ Page 4 extracted at 400 DPI!")
    print(f"   Image size: {image_bgr.shape[1]} x {image_bgr.shape[0]} pixels")
    
    # Display original image
    plt.figure(figsize=(12, 16))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title("Original Page 4 (400 DPI)", fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## Step 5: Enhanced Preprocessing with 8 Variants

In [ ]:
print("\n🔧 Enhanced Preprocessing for HIGH ACCURACY OCR...\n")

# Create multiple preprocessing variants
preprocessing_variants = {}

# --- VARIANT 1: Original grayscale (high quality) ---
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

# Histogram stretching for better contrast
p2, p98 = np.percentile(gray, (2, 98))
if p98 - p2 > 0:
    gray_stretched = cv2.convertScaleAbs(gray, alpha=255.0/(p98-p2), beta=-p2*255.0/(p98-p2))
else:
    gray_stretched = gray.copy()
gray_stretched = np.clip(gray_stretched, 0, 255).astype(np.uint8)
preprocessing_variants['grayscale'] = gray_stretched
print("  ✓ Variant 1: Grayscale stretched")

# --- VARIANT 2: Otsu's thresholding ---
blur = cv2.GaussianBlur(gray_stretched, (3, 3), 0)
_, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
preprocessing_variants['otsu'] = otsu
print("  ✓ Variant 2: Otsu's threshold")

# --- VARIANT 3: Adaptive threshold (small block) ---
adaptive_11 = cv2.adaptiveThreshold(
    gray_stretched, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
    cv2.THRESH_BINARY, 11, 2
)
preprocessing_variants['adaptive_11'] = adaptive_11
print("  ✓ Variant 3: Adaptive threshold (11px blocks)")

# --- VARIANT 4: Adaptive threshold (medium block) ---
adaptive_21 = cv2.adaptiveThreshold(
    gray_stretched, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
    cv2.THRESH_BINARY, 21, 4
)
preprocessing_variants['adaptive_21'] = adaptive_21
print("  ✓ Variant 4: Adaptive threshold (21px blocks)")

# --- VARIANT 5: Sharpened ---
sharpen_kernel = np.array([[0,-1,0], [-1,5,-1], [0,-1,0]])
sharpened = cv2.filter2D(gray_stretched, -1, sharpen_kernel)
preprocessing_variants['sharpened'] = sharpened
print("  ✓ Variant 5: Sharpened")

# --- VARIANT 6: CLAHE ---
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
clahe_img = clahe.apply(gray)
preprocessing_variants['clahe'] = clahe_img
print("  ✓ Variant 6: CLAHE enhanced")

# --- VARIANT 7: Denoised ---
denoised = cv2.bilateralFilter(gray, 11, 75, 75)
preprocessing_variants['denoised'] = denoised
print("  ✓ Variant 7: Bilateral denoised")

# --- VARIANT 8: Morphologically cleaned ---
kernel = np.ones((1, 1), np.uint8)
morph = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, kernel)
preprocessing_variants['morphological'] = morph
print("  ✓ Variant 8: Morphological cleaned")

# For checkbox detection, use Otsu
processed = otsu

print("\n✅ Created 8 preprocessing variants for optimal OCR!")

# Display preprocessing results
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Preprocessing Variants for Best OCR', fontsize=14, fontweight='bold')

for idx, (name, img) in enumerate(preprocessing_variants.items()):
    r, c = idx // 4, idx % 4
    axes[r, c].imshow(img, cmap='gray')
    axes[r, c].set_title(name, fontsize=10)
    axes[r, c].axis('off')

plt.tight_layout()
plt.show()

## Step 6: Strict Checkbox Detection

In [ ]:
print("\n☑️  Detecting checkboxes with STRICT filtering...\n")

checkbox_image = image_bgr.copy()

# Find contours on inverted Otsu image
inverted = cv2.bitwise_not(preprocessing_variants['otsu'])
contours, _ = cv2.findContours(inverted, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

print(f"  Found {len(contours)} contours, filtering...")

checkbox_locations = []

# VERY STRICT parameters
MIN_SIZE = 18
MAX_SIZE = 55
ASPECT_MIN = 0.75
ASPECT_MAX = 1.35

for contour in contours:
    x, y, w, h = cv2.boundingRect(contour)
    
    # Size check
    if not (MIN_SIZE <= w <= MAX_SIZE and MIN_SIZE <= h <= MAX_SIZE):
        continue
    
    # Aspect ratio check (must be square-ish)
    aspect = float(w) / h
    if not (ASPECT_MIN <= aspect <= ASPECT_MAX):
        continue
    
    # Contour area vs bounding box
    area = cv2.contourArea(contour)
    bbox_area = w * h
    fill = area / bbox_area if bbox_area > 0 else 0
    
    # Checkboxes: moderate fill (outline or checked)
    if fill < 0.15 or fill > 0.92:
        continue
    
    # Polygon approximation
    eps = 0.05 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, eps, True)
    
    # Should have 4-6 vertices
    if 4 <= len(approx) <= 6:
        checkbox_locations.append((x, y, w, h))

# Remove duplicates
unique_boxes = []
for box in checkbox_locations:
    x1, y1, w1, h1 = box
    is_dup = False
    for ux, uy, uw, uh in unique_boxes:
        if abs(x1 - ux) < 12 and abs(y1 - uy) < 12:
            is_dup = True
            break
    if not is_dup:
        unique_boxes.append(box)

checkbox_locations = unique_boxes
checkbox_count = len(checkbox_locations)

# Sort and draw
checkbox_locations.sort(key=lambda b: (b[1] // 40, b[0]))

for i, (x, y, w, h) in enumerate(checkbox_locations, 1):
    cv2.rectangle(checkbox_image, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(checkbox_image, str(i), (x + 2, y - 3), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

print(f"✅ Detected {checkbox_count} checkboxes")

plt.figure(figsize=(14, 18))
plt.imshow(cv2.cvtColor(checkbox_image, cv2.COLOR_BGR2RGB))
plt.title(f'Detected Checkboxes: {checkbox_count}', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## Step 7: High-Accuracy Multi-Strategy OCR Extraction

In [ ]:
print("\n📝 HIGH-ACCURACY OCR Extraction with Multi-Strategy Testing...\n")

# Tesseract configurations to test
configs = {
    'best': r'--oem 3 --psm 6 -c preserve_interword_spaces=1',
    'psm3': r'--oem 3 --psm 3',
    'psm4': r'--oem 3 --psm 4',
    'legacy': r'--oem 1 --psm 6',
}

# Test each variant with each config
best_score = 0
best_text = ""
best_variant = ""
best_config = ""
best_conf = 0

print("  Testing preprocessing variants with multiple Tesseract configs...")

variants_to_test = ['grayscale', 'sharpened', 'clahe', 'denoised', 'adaptive_11']
total_tests = len(variants_to_test) * len(configs)
current_test = 0

for variant_name in variants_to_test:
    img = preprocessing_variants[variant_name]
    
    for cfg_name, cfg in configs.items():
        current_test += 1
        
        try:
            # Get OCR with confidence data
            data = pytesseract.image_to_data(
                img, output_type=pytesseract.Output.DICT, config=cfg
            )
            
            # Calculate confidence (only for real text)
            confs = [int(c) for i, c in enumerate(data['conf']) 
                     if str(c) != '-1' and int(c) > 0 
                     and len(str(data['text'][i]).strip()) > 0]
            
            if len(confs) > 20:  # Need enough data points
                avg_conf = sum(confs) / len(confs)
                text = pytesseract.image_to_string(img, config=cfg)
                text_len = len(text.strip())
                
                # Score: confidence + text length bonus
                score = avg_conf + min(text_len / 100, 20)
                
                if score > best_score:
                    best_score = score
                    best_text = text
                    best_variant = variant_name
                    best_config = cfg_name
                    best_conf = avg_conf
                    
        except Exception as e:
            pass
        
        # Progress indicator
        if current_test % 5 == 0:
            print(f"  Progress: {current_test}/{total_tests} combinations tested...")

print(f"\n  ✓ Best variant: {best_variant}")
print(f"  ✓ Best config: {best_config}")
print(f"  ✓ Base confidence: {best_conf:.1f}%")

# Final extraction with best settings
final_img = preprocessing_variants[best_variant]
final_cfg = configs[best_config]

extracted_text = pytesseract.image_to_string(final_img, config=final_cfg)

# Get real confidence
data = pytesseract.image_to_data(final_img, output_type=pytesseract.Output.DICT, config=final_cfg)
real_confs = [int(c) for i, c in enumerate(data['conf']) 
              if str(c) != '-1' and int(c) > 0 
              and len(str(data['text'][i]).strip()) > 0]

avg_confidence = sum(real_confs) / len(real_confs) if real_confs else 0

# Try to recover missing text from other variants
print("\n  Checking for missed text using alternate variants...")

key_fields = ['legal description', 'purchase transaction', 'fha', 'va case', 'ac e1/2', 'nw1/4']

for alt in ['sharpened', 'clahe', 'adaptive_11', 'adaptive_21']:
    if alt != best_variant:
        alt_text = pytesseract.image_to_string(
            preprocessing_variants[alt], 
            config=r'--oem 3 --psm 6 -c preserve_interword_spaces=1'
        )
        # Add unique lines that contain key fields
        for line in alt_text.split('\n'):
            line = line.strip()
            if len(line) > 15 and line not in extracted_text:
                # Check if this adds new info
                if any(kw in line.lower() for kw in key_fields):
                    extracted_text += "\n" + line
                    print(f"    + Added: {line[:50]}...")

print(f"\n✅ OCR Extraction Complete!")
print(f"   Best preprocessing: {best_variant}")
print(f"   Best Tesseract config: {best_config}")
print(f"   Final confidence: {avg_confidence:.1f}%")
print(f"   Total characters: {len(extracted_text):,}")

## Step 8: Calculate Statistics

In [ ]:
print("\n🔢 Calculating statistics...\n")

# Count words
words = extracted_text.split()
word_count = len(words)

# Count lines (non-empty)
lines = [l for l in extracted_text.split('\n') if l.strip()]
line_count = len(lines)

# Count characters (excluding whitespace)
char_count = len(extracted_text.replace(' ', '').replace('\n', ''))

print(f"✅ Statistics:")
print(f"   Total words: {word_count:,}")
print(f"   Total lines: {line_count}")
print(f"   Total characters: {char_count:,}")
print(f"   Total checkboxes: {checkbox_count}")
print(f"   OCR confidence: {avg_confidence:.1f}%")

## Step 9: Print Complete Summary

In [ ]:
print("\n" + "=" * 80)
print(" " * 20 + "PDF OCR EXTRACTION SUMMARY (v3 - High Accuracy)")
print("=" * 80)
print(f"\nPDF File: {pdf_filename}")
print(f"Page Analyzed: 4")
print(f"Extraction Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Best Preprocessing: {best_variant}")
print(f"Best OCR Config: {best_config}")
print("\n" + "-" * 80)
print("STATISTICS")
print("-" * 80)
print(f"Total Words:      {word_count:,}")
print(f"Total Lines:      {line_count:,}")
print(f"Total Characters: {char_count:,}")
print(f"Total Checkboxes: {checkbox_count:,}")
print(f"OCR Confidence:   {avg_confidence:.1f}%")
print("\n" + "-" * 80)
print("EXTRACTED TEXT")
print("-" * 80)
print(extracted_text)
print("\n" + "=" * 80)
print(" " * 25 + "END OF SUMMARY")
print("=" * 80 + "\n")

## Step 10: Generate and Download Text File (.txt)

In [ ]:
print("\n📄 Generating text file...\n")

# Create text file content
txt_filename = f"page4_extraction_v3_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

txt_content = f"""PDF OCR EXTRACTION REPORT (v3 - High Accuracy)
{'=' * 80}

Source PDF: {pdf_filename}
Page Number: 4
Extraction Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Best Preprocessing: {best_variant}
Best OCR Config: {best_config}

{'=' * 80}
STATISTICS
{'=' * 80}

Total Words:      {word_count:,}
Total Lines:      {line_count:,}
Total Characters: {char_count:,}
Total Checkboxes: {checkbox_count:,}
OCR Confidence:   {avg_confidence:.1f}%

{'=' * 80}
EXTRACTED TEXT
{'=' * 80}

{extracted_text}

{'=' * 80}
END OF REPORT
{'=' * 80}
"""

# Write to file
with open(txt_filename, 'w', encoding='utf-8') as f:
    f.write(txt_content)

print(f"✅ Text file created: {txt_filename}")
print(f"   File size: {os.path.getsize(txt_filename):,} bytes")

# Download the file
print("\n⬇️  Downloading text file...")
files.download(txt_filename)
print("✅ Download started!")

## Step 11: Generate and Download Word File (.docx)

In [ ]:
print("\n📘 Generating Word document...\n")

# Create Word document
docx_filename = f"page4_extraction_v3_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"
doc = Document()

# Add title
title = doc.add_heading('PDF OCR Extraction Report (v3 - High Accuracy)', 0)
title.alignment = 1  # Center alignment

# Add metadata section
doc.add_heading('Document Information', 1)
info_table = doc.add_table(rows=5, cols=2)
info_table.style = 'Light Grid Accent 1'

info_data = [
    ('Source PDF:', pdf_filename),
    ('Page Number:', '4'),
    ('Extraction Date:', datetime.now().strftime('%Y-%m-%d %H:%M:%S')),
    ('Best Preprocessing:', best_variant),
    ('Best OCR Config:', best_config)
]

for idx, (key, value) in enumerate(info_data):
    row = info_table.rows[idx]
    row.cells[0].text = key
    row.cells[1].text = str(value)

doc.add_paragraph()  # Spacing

# Add statistics section
doc.add_heading('Extraction Statistics', 1)
stats_table = doc.add_table(rows=5, cols=2)
stats_table.style = 'Light Grid Accent 1'

stats_data = [
    ('Total Words:', f"{word_count:,}"),
    ('Total Lines:', f"{line_count:,}"),
    ('Total Characters:', f"{char_count:,}"),
    ('Total Checkboxes:', f"{checkbox_count:,}"),
    ('OCR Confidence:', f"{avg_confidence:.1f}%")
]

for idx, (key, value) in enumerate(stats_data):
    row = stats_table.rows[idx]
    row.cells[0].text = key
    row.cells[1].text = value

doc.add_paragraph()  # Spacing

# Add extracted text section
doc.add_heading('Extracted Text', 1)
text_para = doc.add_paragraph(extracted_text)
text_para.style = 'Normal'

# Add footer
doc.add_paragraph()  # Spacing
footer = doc.add_paragraph('\n' + '=' * 80)
footer.add_run('\nGenerated by PDF OCR Extractor v3 - High Accuracy Edition').bold = True
footer.alignment = 1  # Center alignment

# Save document
doc.save(docx_filename)

print(f"✅ Word document created: {docx_filename}")
print(f"   File size: {os.path.getsize(docx_filename):,} bytes")

# Download the file
print("\n⬇️  Downloading Word document...")
files.download(docx_filename)
print("✅ Download started!")

## Step 12: Final Summary

In [ ]:
print("\n" + "*" * 80)
print(" " * 20 + "✅ ALL TASKS COMPLETED! (v3 High Accuracy)")
print("*" * 80)
print("\nCompleted Tasks:")
print("  ✓ PDF uploaded and page 4 extracted (400 DPI)")
print("  ✓ Enhanced preprocessing with 8 variants")
print("  ✓ Strict checkbox detection")
print("  ✓ Multi-strategy OCR extraction")
print("  ✓ Fallback extraction for missed fields")
print("  ✓ Statistics calculated")
print("  ✓ Text file (.txt) generated and downloaded")
print("  ✓ Word file (.docx) generated and downloaded")
print("\nOutput Files:")
print(f"  📄 {txt_filename}")
print(f"  📘 {docx_filename}")
print("\nKey Results:")
print(f"  📊 Words: {word_count:,}")
print(f"  ☑️  Checkboxes: {checkbox_count}")
print(f"  ⭐ Confidence: {avg_confidence:.1f}%")
print(f"  🔧 Best variant: {best_variant}")
print(f"  ⚙️  Best config: {best_config}")
print("\n" + "*" * 80 + "\n")